# 2D Darcy flux: mass balance over a control volume

A steady two-dimensional Darcy flux field (cm/hr) passes through a 1 cm cube of porous media. The question is how fast the
volumetric water content $\theta$ inside the cube changes. It is answered three ways, each more faithful to the field than the last:

1. **Point divergence**: evaluate $-\nabla\cdot\mathbf q$ at the cube's centroid.
2. **Face-center flux**: approximate each face's flow with the flux at its center (midpoint rule).
3. **Exact face integrals**: integrate the normal flux over each face analytically, and cross-check with the divergence theorem.

Every headline number is checked with an `assert`, so if this notebook runs top to bottom, the answers are right.

## 1. Setup

In [ ]:
import json
import math
import uuid
from fractions import Fraction

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import scipy
from scipy.integrate import quad, solve_ivp
from IPython.display import HTML, display

import riskplot
from riskplot import PlotConfig, SurfaceRiskPlot

%matplotlib inline
%config InlineBackend.figure_format = "retina"
np.set_printoptions(linewidth=120, precision=2, suppress=True, floatmode="fixed")

for mod in (np, pd, scipy, matplotlib, riskplot):
    print(f"{mod.__name__:<11} {mod.__version__}")

IN_COLOR, OUT_COLOR, NET_COLOR = "#2166ac", "#b2182b", "#4d4d4d"
INK, MUTED, RULE = "#2a3f5f", "#667a95", "#e3e9f2"

display(HTML(f'''
<style>
  .jp-RenderedImage img, .output_png img, .jp-OutputArea-output > img {{
    display: block; margin: 0 auto; max-width: 100%; height: auto;
  }}
  .jp-RenderedImage, div[id^="view3d-"] {{
    margin: 2.6rem auto; padding: 18px 16px 10px; background: #fff;
    border: 1px solid {RULE}; border-radius: 14px;
    box-shadow: 0 1px 2px rgba(16, 24, 40, 0.04), 0 10px 28px rgba(16, 24, 40, 0.06);
  }}
  .jp-RenderedHTMLCommon h2 {{ margin-top: 2.8rem; color: {INK}; letter-spacing: -0.01em; }}
  .jp-RenderedHTMLCommon p {{ line-height: 1.65; }}
</style>
'''))

plt.rcParams.update({
    "figure.dpi": 110, "savefig.bbox": "tight",
    "font.size": 11, "axes.titlesize": 15, "axes.titlepad": 14, "axes.labelsize": 12,
    "axes.titlecolor": INK, "axes.labelcolor": INK, "axes.edgecolor": RULE, "axes.linewidth": 1.0,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK, "ytick.labelcolor": INK,
    "legend.frameon": False, "figure.facecolor": "white",
})


def check(name, value, expected, tol):
    assert math.isclose(float(value), expected, rel_tol=0.0, abs_tol=tol), (
        f"{name}: got {float(value)!r}, expected {expected!r}"
    )
    print(f"  ok  {name:<34} {float(value):>12.4f}")

## 2. The flux field and its divergence

The flux has no $z$-component and no $z$-dependence:

$$
q_x(x,y) = 3x^2 + 5xy + 7y^2, \qquad
q_y(x,y) = -2x^3 + 4x^2y^3 - 2y \qquad [\mathrm{cm/hr}]
$$

With no sources or sinks, continuity (conservation of water volume) says storage changes only by net inflow:

$$
\frac{\partial \theta}{\partial t} = -\nabla\cdot\mathbf q = -\left(\frac{\partial q_x}{\partial x} + \frac{\partial q_y}{\partial y}\right),
\qquad
\frac{\partial q_x}{\partial x} = 6x + 5y,
\qquad
\frac{\partial q_y}{\partial y} = 12x^2y^2 - 2 .
$$

Each polynomial is stored as its coefficients `{(i, j): c}`, meaning $c\,x^i y^j$. The same objects can be evaluated on
numpy grids *and* differentiated or integrated exactly in rational arithmetic, so the derivatives above are computed, not typed in.

In [ ]:
QX = {(2, 0): 3, (1, 1): 5, (0, 2): 7}      # 3x^2 + 5xy + 7y^2
QY = {(3, 0): -2, (2, 3): 4, (0, 1): -2}    # -2x^3 + 4x^2 y^3 - 2y


def evaluate(poly, x, y):
    return sum(c * x**i * y**j for (i, j), c in poly.items())


def d_dx(poly):
    return {(i - 1, j): c * i for (i, j), c in poly.items() if i > 0}


def d_dy(poly):
    return {(i, j - 1): c * j for (i, j), c in poly.items() if j > 0}


DQX_DX = d_dx(QX)
DQY_DY = d_dy(QY)
assert DQX_DX == {(1, 0): 6, (0, 1): 5}        # 6x + 5y
assert DQY_DY == {(2, 2): 12, (0, 0): -2}      # 12x^2 y^2 - 2


def qx(x, y):
    return evaluate(QX, x, y)


def qy(x, y):
    return evaluate(QY, x, y)


def div_q(x, y):
    return evaluate(DQX_DX, x, y) + evaluate(DQY_DY, x, y)


rng = np.random.default_rng(0)
pts = rng.uniform([1.5, 2.5], [3.5, 4.5], size=(20, 2))
eps = 1e-6
fd_div = np.array([(qx(x + eps, y) - qx(x - eps, y)) / (2 * eps)
                   + (qy(x, y + eps) - qy(x, y - eps)) / (2 * eps) for x, y in pts])
assert np.allclose(fd_div, div_q(pts[:, 0], pts[:, 1]), rtol=1e-7)

X0, X1, Y0, Y1, DZ = 2, 3, 3, 4, 1
XC, YC = (X0 + X1) / 2, (Y0 + Y1) / 2
V = (X1 - X0) * (Y1 - Y0) * DZ
A_X = (Y1 - Y0) * DZ      # area of the left/right faces
A_Y = (X1 - X0) * DZ      # area of the bottom/top faces
print(f"V = {V} cm^3, face areas = {A_X}, {A_Y} cm^2, centroid = ({XC}, {YC})")

## 3. The control volume in 3D

**Control volume in $xyz$ space** — the 1 cm cube sitting in the flux field, with every face carrying its own flow. Blue
faces ($x=2$ and $y=3$) take water in and red faces ($x=3$ and $y=4$) let it out; cones point the way the water moves and
grow with $\log_{10}|\mathbf q|$. The $z=0$ and $z=1$ faces are left open because $q_z = 0$, so nothing crosses them. Grey
lines are streamlines through the mid-plane $z = 0.5$.

Drag to rotate, pinch or scroll to zoom, double-click to reset, hover a face or cone for its flux.

In [ ]:
# A depth-sorted 3D viewer drawn in SVG: RiskPlot has no 3D plot, and WebGL leaves the
# scene blank in some browsers. A scene is a dict of polygons, polylines, cones and labels.
VIEW3D_JS = '''
(function () {
  if (window.View3D) return;
  function esc(s) {
    return String(s).replace(/&/g, "&amp;").replace(/</g, "&lt;").replace(/>/g, "&gt;").replace(/"/g, "&quot;");
  }
  function cross(a, b) { return [a[1] * b[2] - a[2] * b[1], a[2] * b[0] - a[0] * b[2], a[0] * b[1] - a[1] * b[0]]; }
  function unit(a) { var m = Math.hypot(a[0], a[1], a[2]) || 1; return [a[0] / m, a[1] / m, a[2] / m]; }
  function hull(pts) {
    pts = pts.slice().sort(function (a, b) { return a[0] - b[0] || a[1] - b[1]; });
    function cr(o, a, b) { return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0]); }
    var lo = [], up = [], i;
    for (i = 0; i < pts.length; i++) {
      while (lo.length >= 2 && cr(lo[lo.length - 2], lo[lo.length - 1], pts[i]) <= 0) lo.pop();
      lo.push(pts[i]);
    }
    for (i = pts.length - 1; i >= 0; i--) {
      while (up.length >= 2 && cr(up[up.length - 2], up[up.length - 1], pts[i]) <= 0) up.pop();
      up.push(pts[i]);
    }
    lo.pop(); up.pop();
    return lo.concat(up);
  }
  function ptsAttr(pr) {
    var s = "";
    for (var i = 0; i < pr.length; i++) s += (i ? " " : "") + pr[i][0].toFixed(1) + "," + pr[i][1].toFixed(1);
    return s;
  }

  window.View3D = function (root, S) {
    var W = S.width, H = S.height, B = S.bounds, A = S.aspect;
    var UP = S.up === "z" ? 2 : 1;
    var mid = [0, 1, 2].map(function (i) { return (B[i][0] + B[i][1]) / 2; });
    var hr = [0, 1, 2].map(function (i) { return (B[i][1] - B[i][0]) / 2; });
    function norm(p) { return [0, 1, 2].map(function (i) { return (p[i] - mid[i]) / hr[i] * A[i]; }); }
    function toView(n) { return UP === 2 ? [n[0], n[2], -n[1]] : [n[0], n[1], n[2]]; }
    function V(p) { return toView(norm(p)); }

    var cam = { yaw: S.camera[0], pitch: S.camera[1], zoom: 1 }, cy, sy, cp, sp, scale, DIST = 6;
    function setCam() {
      cy = Math.cos(cam.yaw); sy = Math.sin(cam.yaw); cp = Math.cos(cam.pitch); sp = Math.sin(cam.pitch);
      scale = Math.min(W, H) * 0.33 * cam.zoom;
    }
    function rot(v) {
      var x1 = v[0] * cy + v[2] * sy, z1 = -v[0] * sy + v[2] * cy;
      return [x1, v[1] * cp - z1 * sp, v[1] * sp + z1 * cp];
    }
    function project(v) {
      var r = rot(v), f = DIST / (DIST - r[2]);
      return [W / 2 + scale * r[0] * f, H / 2 - scale * r[1] * f, r[2]];
    }

    root.style.position = "relative";
    root.style.maxWidth = W + "px";
    root.style.fontFamily = "-apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif";
    root.innerHTML =
      (S.title ? '<div style="font-size:18px;font-weight:600;letter-spacing:-0.01em;color:#2a3f5f;margin:0 0 2px 4px">' + esc(S.title) + "</div>" : "") +
      (S.subtitle ? '<div style="font-size:12.5px;color:#667a95;margin:0 0 6px 4px">' + esc(S.subtitle) + "</div>" : "") +
      '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 ' + W + " " + H + '" width="100%" ' +
      'style="display:block;touch-action:none;cursor:grab;user-select:none;-webkit-user-select:none"></svg>' +
      '<div style="font-size:12px;color:#8c9bb0;text-align:center;margin-top:2px">Drag to rotate &middot; pinch or click then scroll to zoom &middot; double-click to reset</div>' +
      '<div class="v3d-legend" style="display:flex;flex-wrap:wrap;justify-content:center;gap:6px 20px;font-size:13px;color:#2a3f5f;margin:10px 8px 4px"></div>' +
      '<div class="v3d-cbar"></div>' +
      '<div class="v3d-tip" style="display:none;position:absolute;pointer-events:none;background:rgba(255,255,255,0.95);' +
      'border:1px solid #c8d4e3;border-radius:4px;padding:4px 8px;font-size:12px;color:#2a3f5f;white-space:nowrap"></div>';
    var svg = root.querySelector("svg"), tip = root.querySelector(".v3d-tip");

    (S.legend || []).forEach(function (L) {
      var sw = L.kind === "line"
        ? '<svg width="26" height="10"><line x1="1" y1="5" x2="25" y2="5" stroke="' + L.c + '" stroke-width="' + (L.w || 3) + '"/></svg>'
        : L.kind === "cone"
          ? '<svg width="16" height="14"><polygon points="8,1 14,13 2,13" fill="' + L.c + '"/></svg>'
          : '<span style="display:inline-block;width:14px;height:14px;background:' + L.c + ';opacity:' + (L.op || 0.45) + ';border:1px solid ' + L.c + '"></span>';
      root.querySelector(".v3d-legend").insertAdjacentHTML("beforeend",
        '<span style="display:inline-flex;align-items:center;gap:6px">' + sw + esc(L.label) + "</span>");
    });
    if (S.colorbar) {
      var C = S.colorbar, ticks = C.ticks.map(function (t) {
        var pct = (t - C.min) / (C.max - C.min) * 100;
        return '<span style="position:absolute;left:' + pct + '%;transform:translateX(-50%);top:14px">' + t + "</span>";
      }).join("");
      root.querySelector(".v3d-cbar").innerHTML =
        '<div style="display:flex;justify-content:center;align-items:flex-start;gap:10px;font-size:12px;color:#2a3f5f;margin:4px 0 22px">' +
        '<span style="padding-top:0">' + esc(C.title) + '</span><div style="position:relative;width:320px">' +
        '<div style="height:12px;background:' + C.css + '"></div>' + ticks + "</div></div>";
    }

    var polys = (S.polys || []).map(function (P) {
      return { v: P.p.map(V), a: 'fill="' + P.fill + '" fill-opacity="' + (P.op == null ? 1 : P.op) + '" ' +
        (P.stroke ? 'stroke="' + P.stroke + '" stroke-width="' + (P.sw || 1) + '"' : 'stroke="none"') +
        (P.t ? ' data-t="' + esc(P.t) + '"' : "") };
    });
    var segs = [];
    (S.lines || []).forEach(function (L) {
      var v = L.p.map(V), step = L.chunk || 3;
      var a = 'fill="none" stroke="' + L.c + '" stroke-width="' + (L.w || 1) + '" stroke-linecap="round" stroke-linejoin="round"' +
        (L.op == null ? "" : ' stroke-opacity="' + L.op + '"');
      for (var i = 0; i < v.length - 1; i += step) segs.push({ v: v.slice(i, Math.min(i + step + 1, v.length)), a: a, top: !!L.top });
    });
    var cones = (S.cones || []).map(function (K) {
      var tipN = norm(K.tip), baseN = norm(K.base);
      var ax = [tipN[0] - baseN[0], tipN[1] - baseN[1], tipN[2] - baseN[2]];
      var len = Math.hypot(ax[0], ax[1], ax[2]);
      var u = unit(cross(ax, Math.abs(ax[0]) < 0.9 * len ? [1, 0, 0] : [0, 1, 0])), w = unit(cross(ax, u));
      var r = K.r * len, ring = [];
      for (var k = 0; k < 16; k++) {
        var t = 2 * Math.PI * k / 16, c = Math.cos(t) * r, s = Math.sin(t) * r;
        ring.push(toView([baseN[0] + c * u[0] + s * w[0], baseN[1] + c * u[1] + s * w[1], baseN[2] + c * u[2] + s * w[2]]));
      }
      return { tip: toView(tipN), base: toView(baseN), ring: ring, c: K.c, t: K.t ? ' data-t="' + esc(K.t) + '"' : "" };
    });

    function wallCorner(i, s, a, b) {
      var n = [0, 0, 0], j = (i + 1) % 3, k = (i + 2) % 3;
      n[i] = s ? A[i] : -A[i]; n[j] = a ? A[j] : -A[j]; n[k] = b ? A[k] : -A[k];
      return n;
    }
    function isBack(i, s) {
      var n = [0, 0, 0]; n[i] = s ? 1 : -1;
      return rot(toView(n))[2] < 0;
    }
    function nrm(i, value) { return (value - mid[i]) / hr[i] * A[i]; }

    function render() {
      setCam();
      var out = [], i, s, j, k;
      var back = {};
      for (i = 0; i < 3; i++) for (s = 0; s < 2; s++) back[i + "" + s] = isBack(i, s);

      for (i = 0; i < 3; i++) for (s = 0; s < 2; s++) {
        if (!back[i + "" + s]) continue;
        var cs = [[0, 0], [1, 0], [1, 1], [0, 1]].map(function (ab) { return project(toView(wallCorner(i, s, ab[0], ab[1]))); });
        out.push('<polygon points="' + ptsAttr(cs) + '" fill="#e5ecf6" stroke="none"/>');
        [(i + 1) % 3, (i + 2) % 3].forEach(function (g) {
          var o = 3 - i - g;
          (S.ticks[g] || []).forEach(function (tv) {
            var p0 = [0, 0, 0], p1 = [0, 0, 0];
            p0[i] = p1[i] = s ? A[i] : -A[i]; p0[g] = p1[g] = nrm(g, tv); p0[o] = -A[o]; p1[o] = A[o];
            var q0 = project(toView(p0)), q1 = project(toView(p1));
            out.push('<line x1="' + q0[0].toFixed(1) + '" y1="' + q0[1].toFixed(1) + '" x2="' + q1[0].toFixed(1) + '" y2="' + q1[1].toFixed(1) + '" stroke="#fff" stroke-width="1.2"/>');
          });
        });
      }

      var items = [];
      polys.forEach(function (P) {
        var pr = P.v.map(project), d = 0;
        for (var q = 0; q < pr.length; q++) d += pr[q][2];
        items.push([d / pr.length, '<polygon points="' + ptsAttr(pr) + '" ' + P.a + "/>"]);
      });
      var overlay = [];
      segs.forEach(function (G) {
        var pr = G.v.map(project), d = 0;
        for (var q = 0; q < pr.length; q++) d += pr[q][2];
        var html = '<polyline points="' + ptsAttr(pr) + '" ' + G.a + "/>";
        if (G.top) overlay.push(html); else items.push([d / pr.length + 0.001, html]);
      });
      cones.forEach(function (K) {
        var pt = project(K.tip), pb = project(K.base), ring = K.ring.map(project);
        var shape = hull(ring.concat([pt]));
        var html = '<polygon points="' + ptsAttr(shape) + '" fill="' + K.c + '" stroke="rgba(0,0,0,0.35)" stroke-width="0.6"' + K.t + "/>";
        if (pt[2] < pb[2]) {
          html += '<polygon points="' + ptsAttr(ring) + '" fill="' + K.c + '" stroke="rgba(0,0,0,0.35)" stroke-width="0.6"' + K.t + "/>" +
                  '<polygon points="' + ptsAttr(ring) + '" fill="#000" fill-opacity="0.28" stroke="none"' + K.t + "/>";
        }
        items.push([(pt[2] + pb[2]) / 2, html]);
      });
      items.sort(function (a, b) { return a[0] - b[0]; });
      for (i = 0; i < items.length; i++) out.push(items[i][1]);
      out = out.concat(overlay);

      var center = project([0, 0, 0]);
      for (i = 0; i < 3; i++) {
        j = (i + 1) % 3; k = (i + 2) % 3;
        var best = null;
        for (var sj = 0; sj < 2; sj++) for (var sk = 0; sk < 2; sk++) {
          var silhouette = back[j + "" + sj] !== back[k + "" + sk];
          var e0 = [0, 0, 0]; e0[j] = sj ? A[j] : -A[j]; e0[k] = sk ? A[k] : -A[k];
          var m = e0.slice(); m[i] = 0;
          var pm = project(toView(m));
          var score = (silhouette ? 1e6 : 0) + (i === UP ? -pm[0] : pm[1]);
          if (!best || score > best.score) best = { score: score, e0: e0, pm: pm };
        }
        var dir = [best.pm[0] - center[0], best.pm[1] - center[1]], dl = Math.hypot(dir[0], dir[1]) || 1;
        dir = [dir[0] / dl, dir[1] / dl];
        var anchor = dir[0] > 0.35 ? "start" : dir[0] < -0.35 ? "end" : "middle";
        (S.ticks[i] || []).forEach(function (tv) {
          var e = best.e0.slice(); e[i] = nrm(i, tv);
          var pe = project(toView(e));
          out.push('<text x="' + (pe[0] + dir[0] * 14).toFixed(1) + '" y="' + (pe[1] + dir[1] * 14 + 4).toFixed(1) +
                   '" font-size="12" fill="#2a3f5f" text-anchor="' + anchor + '">' + esc(tv) + "</text>");
        });
        out.push('<text x="' + (best.pm[0] + dir[0] * 44).toFixed(1) + '" y="' + (best.pm[1] + dir[1] * 44 + 5).toFixed(1) +
                 '" font-size="14" fill="#2a3f5f" text-anchor="' + anchor + '">' + esc(S.titles[i]) + "</text>");
      }

      (S.labels || []).forEach(function (L) {
        var pl = project(V(L.p));
        out.push('<text x="' + pl[0].toFixed(1) + '" y="' + pl[1].toFixed(1) + '" font-size="14" font-weight="600" fill="' + L.c +
                 '" text-anchor="middle" stroke="#fff" stroke-width="4" paint-order="stroke">' + esc(L.text) + "</text>");
      });
      svg.innerHTML = out.join("");
      // Grow-only frame: rotating can never clip the scene.
      var now = Date.now();
      if (!frame || now - lastFit > 250) {
        lastFit = now;
        var bb = svg.getBBox(), pad = 0.03 * Math.max(bb.width, bb.height);
        var box = [bb.x - pad, bb.y - pad, bb.x + bb.width + pad, bb.y + bb.height + pad];
        if (frame) {
          box = [Math.min(frame[0], box[0]), Math.min(frame[1], box[1]),
                 Math.max(frame[2], box[2]), Math.max(frame[3], box[3])];
        }
        if (!frame || box.some(function (v, k) { return Math.abs(v - frame[k]) > 0.5; })) {
          frame = box;
          svg.setAttribute("viewBox", [box[0], box[1], box[2] - box[0], box[3] - box[1]].join(" "));
        }
      }
    }

    var frame = null, lastFit = 0;
    var pending = false;
    function schedule() {
      if (pending) return;
      pending = true;
      requestAnimationFrame(function () { pending = false; render(); });
    }
    var drag = null, active = false;
    svg.addEventListener("pointerdown", function (e) {
      drag = { x: e.clientX, y: e.clientY }; active = true;
      svg.setPointerCapture(e.pointerId); svg.style.cursor = "grabbing"; tip.style.display = "none";
    });
    svg.addEventListener("pointermove", function (e) {
      if (drag) {
        cam.yaw += (e.clientX - drag.x) * 0.008;
        cam.pitch = Math.max(-1.5, Math.min(1.5, cam.pitch + (e.clientY - drag.y) * 0.008));
        drag = { x: e.clientX, y: e.clientY };
        schedule();
        return;
      }
      var t = e.target.getAttribute && e.target.getAttribute("data-t");
      if (!t) { tip.style.display = "none"; return; }
      var box = root.getBoundingClientRect();
      tip.innerHTML = t.split("|").map(esc).join("<br>");
      tip.style.display = "block";
      tip.style.left = (e.clientX - box.left + 14) + "px";
      tip.style.top = (e.clientY - box.top + 14) + "px";
    });
    function endDrag() { drag = null; svg.style.cursor = "grab"; lastFit = 0; schedule(); }
    svg.addEventListener("pointerup", endDrag);
    svg.addEventListener("pointercancel", endDrag);
    svg.addEventListener("pointerleave", function () { active = false; tip.style.display = "none"; });
    svg.addEventListener("wheel", function (e) {
      if (!(e.ctrlKey || active)) return;
      e.preventDefault();
      cam.zoom = Math.max(0.5, Math.min(4, cam.zoom * Math.exp(-e.deltaY * 0.002)));
      schedule();
    }, { passive: false });
    svg.addEventListener("dblclick", function () {
      cam.yaw = S.camera[0]; cam.pitch = S.camera[1]; cam.zoom = 1; schedule();
    });
    svg.addEventListener("gesturestart", function (e) { e.preventDefault(); cam.zoom0 = cam.zoom; });
    svg.addEventListener("gesturechange", function (e) {
      e.preventDefault(); cam.zoom = Math.max(0.5, Math.min(4, cam.zoom0 * e.scale)); schedule();
    });
    render();
  };
})();
'''


def _rounded(value):
    if isinstance(value, dict):
        return {k: _rounded(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, np.ndarray)):
        return [_rounded(v) for v in value]
    if isinstance(value, (float, np.floating)):
        return round(float(value), 4)
    if isinstance(value, np.integer):
        return int(value)
    return value


def show_3d(scene):
    div_id = f"view3d-{uuid.uuid4().hex}"
    display(HTML(
        f'<script>{VIEW3D_JS}</script><div id="{div_id}"></div>'
        f'<script>(function () {{ var el = document.getElementById("{div_id}");'
        f' try {{ View3D(el, {json.dumps(_rounded(scene))}); }}'
        f' catch (err) {{ el.textContent = "3D view failed to draw: " + err; }} }})();</script>'
    ))

In [ ]:
Z0, Z1 = 0, DZ
X_WIN, Y_WIN = (1.5, 3.5), (2.5, 4.5)

_gx, _gy = np.meshgrid(np.linspace(*X_WIN, 81), np.linspace(*Y_WIN, 81))
_logmag = np.log10(np.hypot(qx(_gx, _gy), qy(_gx, _gy)))
LOG_LO, LOG_HI = _logmag.min(), _logmag.max()
CONE_MAX = 0.28   # cm


def cone_vector(x, y):
    u, v = qx(x, y), qy(x, y)
    mag = math.hypot(u, v)
    length = CONE_MAX * (math.log10(mag) - LOG_LO + 0.3) / (LOG_HI - LOG_LO + 0.3)
    return np.array([length * u / mag, length * v / mag, 0.0])


zc_ = (Z0 + Z1) / 2
face_specs = [
    ([(X0, Y0, Z0), (X0, Y1, Z0), (X0, Y1, Z1), (X0, Y0, Z1)], IN_COLOR,
     f"left face, x = {X0}|inflow|q_x at center = {qx(X0, YC):.2f} cm/hr"),
    ([(X0, Y0, Z0), (X1, Y0, Z0), (X1, Y0, Z1), (X0, Y0, Z1)], IN_COLOR,
     f"bottom face, y = {Y0}|inflow|q_y at center = {qy(XC, Y0):.2f} cm/hr"),
    ([(X1, Y0, Z0), (X1, Y1, Z0), (X1, Y1, Z1), (X1, Y0, Z1)], OUT_COLOR,
     f"right face, x = {X1}|outflow|q_x at center = {qx(X1, YC):.2f} cm/hr"),
    ([(X0, Y1, Z0), (X1, Y1, Z0), (X1, Y1, Z1), (X0, Y1, Z1)], OUT_COLOR,
     f"top face, y = {Y1}|outflow|q_y at center = {qy(XC, Y1):.2f} cm/hr"),
]
polys = [dict(p=corners, fill=color, op=0.28, t=hover) for corners, color, hover in face_specs]

corner = {(i, j, k): (x, y, z) for i, x in enumerate((X0, X1)) for j, y in enumerate((Y0, Y1))
          for k, z in enumerate((Z0, Z1))}
lines = []
for a in corner:
    for axis in range(3):
        if a[axis] == 0:
            b = tuple(1 if n == axis else a[n] for n in range(3))
            lines.append(dict(p=[corner[a], corner[b]], c="#111", w=3, chunk=1))


def streamline(x_start, y_start):
    def rhs(_, p):
        u, v = qx(p[0], p[1]), qy(p[0], p[1])
        m = math.hypot(u, v)
        return [u / m, v / m]

    def leaves_window(_, p):
        return min(p[0] - X_WIN[0], X_WIN[1] - p[0], p[1] - Y_WIN[0], Y_WIN[1] - p[1])
    leaves_window.terminal = True
    return solve_ivp(rhs, (0, 10), [x_start, y_start], events=leaves_window, max_step=0.05).y


for x_start in np.linspace(1.55, 3.45, 14):
    path = streamline(x_start, Y_WIN[0] + 0.01)
    lines.append(dict(p=[(x, y, zc_) for x, y in zip(*path)], c="#707070", w=2.2))

s3 = [0.2, 0.5, 0.8]
cones = []
for face_pts, color, inflow in [
    ([(X0, Y0 + a, Z0 + c) for a in s3 for c in s3], IN_COLOR, True),
    ([(X0 + a, Y0, Z0 + c) for a in s3 for c in s3], IN_COLOR, True),
    ([(X1, Y0 + a, Z0 + c) for a in s3 for c in s3], OUT_COLOR, False),
    ([(X0 + a, Y1, Z0 + c) for a in s3 for c in s3], OUT_COLOR, False),
]:
    for x, y, z in face_pts:
        p, vec = np.array([x, y, z]), cone_vector(x, y)
        base, tip_ = (p - vec, p) if inflow else (p, p + vec)
        cones.append(dict(tip=tip_, base=base, r=0.32, c=color,
                          t=f"({x:.2f}, {y:.2f}, {z:.2f})|q_x = {qx(x, y):.1f} cm/hr|q_y = {qy(x, y):.1f} cm/hr"))

scene_cv = dict(
    width=900, height=620,
    title="Control volume in xyz space",
    subtitle="z = 0 and z = 1 faces are open: q_z = 0, so no flow crosses them",
    bounds=[X_WIN, Y_WIN, (-0.5, 1.5)], aspect=[1, 1, 1], up="y", camera=[0.5, 0.32],
    ticks=[[1.5, 2, 2.5, 3, 3.5], [2.5, 3, 3.5, 4, 4.5], [0, 0.5, 1]],
    titles=["x [cm]", "y [cm]", "z [cm]"],
    polys=polys, lines=lines, cones=cones,
    labels=[dict(p=(X0 - 0.32, YC, Z1), text="LEFT in", c=IN_COLOR),
            dict(p=(XC, Y0 - 0.3, Z1), text="BOTTOM in", c=IN_COLOR),
            dict(p=(X1 + 0.36, YC, Z1), text="RIGHT out", c=OUT_COLOR),
            dict(p=(XC, Y1 + 0.34, Z1), text="TOP out", c=OUT_COLOR)],
    legend=[dict(label="inflow faces (x = 2, y = 3)", c=IN_COLOR, kind="face"),
            dict(label="outflow faces (x = 3, y = 4)", c=OUT_COLOR, kind="face"),
            dict(label="flux into the box", c=IN_COLOR, kind="cone"),
            dict(label="flux out of the box", c=OUT_COLOR, kind="cone"),
            dict(label="control volume", c="#111", kind="line"),
            dict(label="streamlines (z = 0.5)", c="#707070", kind="line", w=2)],
)
show_3d(scene_cv)

## 4. The field as matrices

`np.meshgrid(x, y)` returns two matrices with `X[i, j] = x[j]` and `Y[i, j] = y[i]`. Any field evaluated on them is a matrix
whose **row $i$ is $y_i$** and **column $j$ is $x_j$**. The coarse 0.5 cm grid below deliberately contains the box corners,
the four face centers, and the centroid. The matrices hold every number Methods 1 and 2 need.

In [ ]:
xc = np.arange(1.5, 3.5 + 1e-9, 0.5)
yc = np.arange(2.5, 4.5 + 1e-9, 0.5)
Xc, Yc = np.meshgrid(xc, yc)

QXc = qx(Xc, Yc)
QYc = qy(Xc, Yc)
MAGc = np.hypot(QXc, QYc)
DIVc = div_q(Xc, Yc)

print("x nodes:", xc)
print("y nodes:", yc)
coarse = {"X": Xc, "Y": Yc, "q_x [cm/hr]": QXc, "q_y [cm/hr]": QYc, "|q| [cm/hr]": MAGc, "div q [1/hr]": DIVc}
for name, M in coarse.items():
    print(f"\n{name}   shape={M.shape}   (row i <-> y[i], column j <-> x[j])")
    print(M)

## 5. The $q_y$ surface

Both remaining figures read from one 161 × 161 sample of the field.

In [ ]:
xf = np.linspace(1.5, 3.5, 161)
yf = np.linspace(2.5, 4.5, 161)
Xf, Yf = np.meshgrid(xf, yf)
QXf, QYf = qx(Xf, Yf), qy(Xf, Yf)
MAGf = np.hypot(QXf, QYf)
LOGf = np.log10(MAGf)
DIVf = div_q(Xf, Yf)

fields = pd.DataFrame({
    "x": Xf.ravel(), "y": Yf.ravel(),
    "q_x": QXf.ravel(), "q_y": QYf.ravel(),
    "log10_mag": LOGf.ravel(), "div_q": DIVf.ravel(),
})
print(f"|q| on the plotting window spans {MAGf.min():.1f} to {MAGf.max():.1f} cm/hr "
      f"({np.log10(MAGf.max() / MAGf.min()):.2f} decades)")
print(f"q_y spans {QYf.min():.1f} to {QYf.max():.1f} cm/hr;  q_x spans {QXf.min():.1f} to {QXf.max():.1f} cm/hr")

**$q_y(x, y)$** — the vertical flux drawn as a height surface, so the $4x^2y^3$ term is visible as a slope. The black curve
traces $q_y$ around the control volume's boundary: it climbs steeply from the bottom face ($y=3$) to the top face ($y=4$),
which is exactly why more water leaves the box than enters it.

Drag to rotate, hover a cell for its value.

In [ ]:
# RiskPlot's interactive surface renders through WebGL, so this uses the SVG viewer instead.
viridis = matplotlib.colormaps["viridis"]
Q_MAX = 4500.0
n_cells = 28
xs = np.linspace(*X_WIN, n_cells + 1)
ys = np.linspace(*Y_WIN, n_cells + 1)
Xs, Ys = np.meshgrid(xs, ys)
Zs = qy(Xs, Ys)

surface_polys = []
for i in range(n_cells):
    for j in range(n_cells):
        quad_pts = [(xs[j], ys[i], Zs[i, j]), (xs[j + 1], ys[i], Zs[i, j + 1]),
                    (xs[j + 1], ys[i + 1], Zs[i + 1, j + 1]), (xs[j], ys[i + 1], Zs[i + 1, j])]
        xm, ym = (xs[j] + xs[j + 1]) / 2, (ys[i] + ys[i + 1]) / 2
        color = matplotlib.colors.to_hex(viridis(float(np.mean(Zs[i:i + 2, j:j + 2])) / Q_MAX))
        surface_polys.append(dict(p=quad_pts, fill=color, stroke=color, sw=0.6,
                                  t=f"x = {xm:.2f}, y = {ym:.2f}|q_y = {qy(xm, ym):.1f} cm/hr"))

s = np.linspace(0, 1, 40)
bx = np.concatenate([X0 + s, np.full_like(s, X1), X1 - s, np.full_like(s, X0)])
by = np.concatenate([np.full_like(s, Y0), Y0 + s, np.full_like(s, Y1), Y1 - s])
boundary = dict(p=list(zip(bx, by, qy(bx, by) + 25)), c="#000", w=3.5, chunk=4, top=True)

stops = ", ".join(f"{matplotlib.colors.to_hex(viridis(t))} {t * 100:.0f}%" for t in np.linspace(0, 1, 9))
scene_qy = dict(
    width=820, height=600,
    title="q_y(x, y) [cm/hr]", subtitle="black curve: q_y along the control volume boundary",
    bounds=[X_WIN, Y_WIN, (0, Q_MAX)], aspect=[1, 1, 0.8], up="z", camera=[-0.55, 0.45],
    ticks=[[1.5, 2, 2.5, 3, 3.5], [2.5, 3, 3.5, 4, 4.5], [0, 1000, 2000, 3000, 4000]],
    titles=["x [cm]", "y [cm]", "q_y [cm/hr]"],
    polys=surface_polys, lines=[boundary],
    legend=[dict(label="control volume boundary", c="#000", kind="line")],
    colorbar=dict(title="q_y [cm/hr]", css=f"linear-gradient(to right, {stops})",
                  min=0, max=Q_MAX, ticks=[0, 1000, 2000, 3000, 4000]),
)
show_3d(scene_qy)

## 6. Streamlines over $\log_{10}|\mathbf q|$

**Streamlines over $\log_{10}|\mathbf q|$** — the flow itself, with the control volume laid over it. White lines follow the
flux; the shading behind them is flux magnitude on a $\log_{10}$ scale, because $|\mathbf q|$ spans more than an order of
magnitude across the window. Streamlines enter through the left and bottom faces and leave through the right and top.

In [ ]:
L_MIN, L_MAX = LOGf.min(), LOGf.max()
def draw_control_volume(ax, color, lw=2.5):
    ax.add_patch(Rectangle((X0, Y0), X1 - X0, Y1 - Y0, fill=False, edgecolor=color, linewidth=lw, zorder=5))
    for (x, y, label, ha, va) in [
        (X0 - 0.04, YC, "left\n(in)", "right", "center"), (X1 + 0.04, YC, "right\n(out)", "left", "center"),
        (XC, Y0 - 0.04, "bottom (in)", "center", "top"), (XC, Y1 + 0.04, "top (out)", "center", "bottom"),
    ]:
        ax.text(x, y, label, ha=ha, va=va, fontsize=9, color=color, fontweight="bold", zorder=6,
                bbox=dict(facecolor="white", alpha=0.75, edgecolor="none", pad=1.5))


cfg = PlotConfig(figsize=(8.0, 7.2), colormap="cividis", grid=False)
fig, ax = SurfaceRiskPlot(cfg).plot(
    fields, "x", "y", "log10_mag", surface_type="contourf", grid_resolution=len(xf),
    title=r"Streamlines over $\log_{10}|\mathbf{q}|$", x_label="x [cm]", y_label="y [cm]",
)
cbar_ax = fig.axes[-1]
cbar_ax.set_ylabel(r"$\log_{10}|\mathbf{q}|$  [cm/hr]", rotation=270, labelpad=18, color=INK)
for spine in cbar_ax.spines.values():
    spine.set_edgecolor(RULE)
# matplotlib: RiskPlot has no vector-field plot.
lw = 0.4 + 2.0 * (LOGf - L_MIN) / (L_MAX - L_MIN)
ax.streamplot(xf, yf, QXf, QYf, color="white", linewidth=lw, density=1.3, arrowsize=1.0)
draw_control_volume(ax, "#d7301f", lw=3)
ax.set_xlim(1.5, 3.5)
ax.set_ylim(2.5, 4.5)
ax.set_aspect("equal")
fig.tight_layout()
plt.show()

## 7. Method 1: point divergence at the centroid

Treat the cube as a single point and evaluate continuity at its centroid $(2.5, 3.5)$:

$$
\frac{\partial\theta}{\partial t}\bigg|_{(2.5,\,3.5)}
= -\big[\,6(2.5) + 5(3.5)\,\big] - \big[\,12(2.5)^2(3.5)^2 - 2\,\big]
= -32.50 - 916.75 = -949.25\ \mathrm{hr^{-1}}
$$

In [ ]:
m1_x = -evaluate(DQX_DX, XC, YC)
m1_y = -evaluate(DQY_DY, XC, YC)
m1_rate = m1_x + m1_y

print("Method 1: point divergence")
check("x-contribution [1/hr]", m1_x, -32.50, 1e-12)
check("y-contribution [1/hr]", m1_y, -916.75, 1e-12)
check("dtheta/dt [1/hr]", m1_rate, -949.25, 1e-12)

## 8. Method 2: face-center flux (midpoint rule)

Take the normal flux at each face's center as representative of the whole face, so $Q_f \approx q_n(\text{center})\,A_f$.
For the control volume, storage changes by net inflow:

$$
V\,\frac{\Delta\theta}{\Delta t} = Q_\text{in} - Q_\text{out}
= \big[q_x(2,3.5) + q_y(2.5,3)\big]A - \big[q_x(3,3.5) + q_y(2.5,4)\big]A
$$

In [ ]:
m2_Q = {
    "left":   qx(X0, YC) * A_X,
    "bottom": qy(XC, Y0) * A_Y,
    "right":  qx(X1, YC) * A_X,
    "top":    qy(XC, Y1) * A_Y,
}
m2_in = m2_Q["left"] + m2_Q["bottom"]
m2_out = m2_Q["right"] + m2_Q["top"]
m2_rate = (m2_in - m2_out) / V
m2_x = (m2_Q["left"] - m2_Q["right"]) / V
m2_y = (m2_Q["bottom"] - m2_Q["top"]) / V

display(pd.DataFrame({"face center (x, y)": [(X0, YC), (XC, Y0), (X1, YC), (XC, Y1)],
                      "q_n [cm/hr]": list(m2_Q.values()),
                      "Q [cm^3/hr]": list(m2_Q.values())}, index=list(m2_Q)).round(4))

print("Method 2: face-center flux")
check("Q_in [cm^3/hr]", m2_in, 770.50, 1e-12)
check("Q_out [cm^3/hr]", m2_out, 1726.00, 1e-12)
check("x-contribution [1/hr]", m2_x, -32.50, 1e-12)
check("dtheta/dt [1/hr]", m2_rate, -955.50, 1e-12)

## 9. Method 3: exact face integrals

Integrate the normal flux over each face ($dz$ integrates to 1 cm). The integrands are polynomials, so the results are exact rationals:

$$
\begin{aligned}
Q_\text{left}   &= \int_3^4 q_x(2,y)\,dy = \int_3^4 (12 + 10y + 7y^2)\,dy = \tfrac{400}{3} \approx 133.3333 \\
Q_\text{bottom} &= \int_2^3 q_y(x,3)\,dx = \int_2^3 (-2x^3 + 108x^2 - 6)\,dx = \tfrac{1291}{2} = 645.5 \\
Q_\text{right}  &= \int_3^4 q_x(3,y)\,dy = \int_3^4 (27 + 15y + 7y^2)\,dy = \tfrac{995}{6} \approx 165.8333 \\
Q_\text{top}    &= \int_2^3 q_y(x,4)\,dx = \int_2^3 (-2x^3 + 256x^2 - 8)\,dx = \tfrac{9485}{6} \approx 1580.8333
\end{aligned}
$$

The **divergence theorem** gives an independent route to the same net outflow:

$$
\oint_{\partial V} \mathbf q\cdot\hat{\mathbf n}\,dA = \int_V \nabla\cdot\mathbf q\,dV
= \int_2^3\!\!\int_3^4 \big(6x + 5y + 12x^2y^2 - 2\big)\,dy\,dx = \tfrac{65}{2} + \tfrac{2806}{3} = \tfrac{5807}{6} \approx 967.8333
$$

Below, the integrals are computed in exact `Fraction` arithmetic. They are asserted equal to the volume integral exactly, and to
`scipy.integrate.quad` numerically, which checks the integrator itself.

In [ ]:
def mono_int(n, a, b):
    a, b = Fraction(a), Fraction(b)
    return (b ** (n + 1) - a ** (n + 1)) / (n + 1)


def face_integral_at_x(poly, x_face, y_lo, y_hi):
    return sum(c * Fraction(x_face) ** i * mono_int(j, y_lo, y_hi) for (i, j), c in poly.items())


def face_integral_at_y(poly, y_face, x_lo, x_hi):
    return sum(c * mono_int(i, x_lo, x_hi) * Fraction(y_face) ** j for (i, j), c in poly.items())


def volume_integral(poly, x_lo, x_hi, y_lo, y_hi, dz=1):
    return dz * sum(c * mono_int(i, x_lo, x_hi) * mono_int(j, y_lo, y_hi) for (i, j), c in poly.items())


m3_Q = {
    "left":   face_integral_at_x(QX, X0, Y0, Y1) * DZ,
    "bottom": face_integral_at_y(QY, Y0, X0, X1) * DZ,
    "right":  face_integral_at_x(QX, X1, Y0, Y1) * DZ,
    "top":    face_integral_at_y(QY, Y1, X0, X1) * DZ,
}
m3_in = m3_Q["left"] + m3_Q["bottom"]
m3_out = m3_Q["right"] + m3_Q["top"]
m3_rate = (m3_in - m3_out) / V
m3_x = (m3_Q["left"] - m3_Q["right"]) / V
m3_y = (m3_Q["bottom"] - m3_Q["top"]) / V
vol_div = volume_integral(DQX_DX, X0, X1, Y0, Y1, DZ) + volume_integral(DQY_DY, X0, X1, Y0, Y1, DZ)

assert m3_Q == {"left": Fraction(400, 3), "bottom": Fraction(1291, 2),
                "right": Fraction(995, 6), "top": Fraction(9485, 6)}
assert vol_div == m3_out - m3_in == Fraction(5807, 6)      # divergence theorem, exactly

quad_Q = {
    "left":   quad(lambda y: qx(X0, y), Y0, Y1)[0] * DZ,
    "bottom": quad(lambda x: qy(x, Y0), X0, X1)[0] * DZ,
    "right":  quad(lambda y: qx(X1, y), Y0, Y1)[0] * DZ,
    "top":    quad(lambda x: qy(x, Y1), X0, X1)[0] * DZ,
}
for face in m3_Q:
    assert math.isclose(quad_Q[face], m3_Q[face], rel_tol=1e-12), face

display(pd.DataFrame({"exact": [str(v) for v in m3_Q.values()],
                      "Q [cm^3/hr]": [float(v) for v in m3_Q.values()],
                      "scipy quad": list(quad_Q.values())}, index=list(m3_Q)).round(4))

print("Method 3: exact face integrals")
check("Q_left [cm^3/hr]", m3_Q["left"], 133.3333, 5e-5)
check("Q_bottom [cm^3/hr]", m3_Q["bottom"], 645.5, 1e-12)
check("Q_right [cm^3/hr]", m3_Q["right"], 165.8333, 5e-5)
check("Q_top [cm^3/hr]", m3_Q["top"], 1580.8333, 5e-5)
check("Q_in [cm^3/hr]", m3_in, 778.8333, 5e-5)
check("Q_out [cm^3/hr]", m3_out, 1746.6667, 5e-5)
check("x-contribution [1/hr]", m3_x, -32.50, 1e-12)
check("dtheta/dt [1/hr]", m3_rate, -967.8333, 5e-5)
check("volume integral of div q", vol_div, 967.8333, 5e-5)
check("div theorem mismatch", vol_div - (m3_out - m3_in), 0.0, 0.0)

## 10. Why the methods disagree

Split each estimate into its $x$ and $y$ parts:

* **$x$: all three agree at −32.50 hr⁻¹.** $\partial q_x/\partial x = 6x + 5y$ is linear, so its average over the box equals its
  centroid value. Likewise $q_x(3,y) - q_x(2,y) = 15 + 5y$ is linear in $y$, so the midpoint rule integrates it exactly.
* **$y$: the $4x^2y^3$ term carries all of the disagreement.**
  * Method 1 → 2. The finite difference of $y^3$ across the box, $(4^3 - 3^3)/1 = 37$, exceeds the point derivative $3y_c^2 = 36.75$.
    That adds $0.25 \times 4x_c^2 = 6.25$.
  * Method 2 → 3. The average of $x^2$ along the top and bottom faces, $\overline{x^2} = x_c^2 + \tfrac{1}{12}$, exceeds $x_c^2$.
    That adds $4 \times 37 \times \tfrac{1}{12} = 12.33$.

In [ ]:
summary = pd.DataFrame(
    {
        "x-part [1/hr]": [m1_x, m2_x, float(m3_x)],
        "y-part [1/hr]": [m1_y, m2_y, float(m3_y)],
        "dtheta/dt [1/hr]": [m1_rate, m2_rate, float(m3_rate)],
        "Q_in [cm^3/hr]": [np.nan, m2_in, float(m3_in)],
        "Q_out [cm^3/hr]": [np.nan, m2_out, float(m3_out)],
    },
    index=["1. point divergence", "2. face-center flux", "3. exact face integrals"],
)
display(summary.round(4))

assert m1_x == m2_x == m3_x == -32.5
check("y-gap, method 1 -> 2", m1_y - m2_y, 6.25, 1e-12)
check("y-gap, method 2 -> 3", m2_y - float(m3_y), 37 / 3, 1e-9)

## 11. Mass balance on the control volume

Inflows add, outflows subtract, and what is left is the net change in stored water, $V\,\partial\theta/\partial t$.

In [ ]:
budget = pd.DataFrame({
    "term": ["left face (in)", "bottom face (in)", "right face (out)", "top face (out)"],
    "Q": [float(m3_Q["left"]), float(m3_Q["bottom"]), -float(m3_Q["right"]), -float(m3_Q["top"])],
})
net = budget["Q"].sum()
display(budget.round(4))
check("budget net = V dtheta/dt", net, -967.8333, 5e-5)

## 12. Convergence as the box shrinks

Shrink the box to half-width $h$ around the centroid (side $2h$). The face-center estimate becomes

$$
\frac{\partial\theta}{\partial t}\bigg|_\text{faces}(h)
= -\frac{q_x(x_c+h,y_c) - q_x(x_c-h,y_c)}{2h} - \frac{q_y(x_c,y_c+h) - q_y(x_c,y_c-h)}{2h}.
$$

These are central differences. They are exact for the quadratic $q_x$, and for $q_y$ only the $y^3$ term contributes error, since
$\frac{(y+h)^3-(y-h)^3}{2h} = 3y^2 + h^2$. So

$$
\frac{\partial\theta}{\partial t}\bigg|_\text{faces}(h) = -949.25 - 4x_c^2h^2 = -949.25 - 25h^2,
$$

which converges to the point divergence at second order. At $h = 0.5$ it recovers −955.50. The exact face integral (volume average of
$-\nabla\cdot\mathbf q$) also converges quadratically, with a larger constant: $-949.25 - 74h^2 - \tfrac{4}{3}h^4$.

In [ ]:
h = np.logspace(np.log10(0.5), -3, 30)


def face_center_rate(h):
    side = 2 * h
    q_in = qx(XC - h, YC) * side * DZ + qy(XC, YC - h) * side * DZ
    q_out = qx(XC + h, YC) * side * DZ + qy(XC, YC + h) * side * DZ
    return (q_in - q_out) / (side * side * DZ)


def exact_rate(h):
    hf, xc_, yc_ = Fraction(h), Fraction(XC), Fraction(YC)
    x_lo, x_hi, y_lo, y_hi = xc_ - hf, xc_ + hf, yc_ - hf, yc_ + hf
    q_in = face_integral_at_x(QX, x_lo, y_lo, y_hi) + face_integral_at_y(QY, y_lo, x_lo, x_hi)
    q_out = face_integral_at_x(QX, x_hi, y_lo, y_hi) + face_integral_at_y(QY, y_hi, x_lo, x_hi)
    return float((q_in - q_out) * DZ / ((x_hi - x_lo) * (y_hi - y_lo) * DZ))


fc = face_center_rate(h)
ex = np.array([exact_rate(v) for v in h])

assert np.allclose(fc, -(949.25 + 25 * h**2), rtol=0, atol=1e-8)
assert np.allclose(ex, -(949.25 + 74 * h**2 + 4 / 3 * h**4), rtol=0, atol=1e-8)
check("face-center rate, h = 0.5", fc[0], -955.50, 1e-9)
check("exact rate, h = 0.5", ex[0], -967.8333, 5e-5)
check("face-center rate, h = 0.001", fc[-1], -949.25, 1e-4)
slope = np.polyfit(np.log(h), np.log(np.abs(fc - m1_rate)), 1)[0]
check("observed convergence order", slope, 2.0, 1e-3)

## 13. Summary

| Method | $\partial\theta/\partial t$ [hr⁻¹] | $Q_\text{in}$ [cm³/hr] | $Q_\text{out}$ [cm³/hr] |
|---|---:|---:|---:|
| 1. Point divergence at (2.5, 3.5) | −949.25 | — | — |
| 2. Face-center flux (midpoint rule) | −955.50 | 770.50 | 1726.00 |
| 3. Exact face integrals | −967.8333 | 778.8333 | 1746.6667 |

All three agree on the $x$-contribution (−32.50 hr⁻¹). The spread comes entirely from the $4x^2y^3$ term in $q_y$. The exact
answer equals the volume integral of $\nabla\cdot\mathbf q$ to the last digit, as the divergence theorem requires.

In [ ]:
print("All assertions passed.")